# Step 0 — build the canonical dataset (CPU runtime, run ONCE)

Runtime > Change runtime type > **CPU**. This uses no GPU quota.

Output: `busi_yolo.zip` + its md5. Every training lane downloads that exact zip.
Nobody ever runs this notebook again.

In [1]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/busi-replication/raw
!pip -q install kaggle opencv-python-headless pyyaml

Mounted at /content/drive


## Get BUSI

**The Kaggle API works without phone verification.** Phone verification gates
notebook internet and accelerators, not dataset downloads via an API token.

Get the token: kaggle.com > your avatar > Settings > API > **Create New Token**.
That downloads `kaggle.json`. Upload it when the next cell prompts.

If the API refuses for any reason, skip to the manual cell below.

In [4]:
!pip -q install --upgrade kaggle

import os, getpass, pathlib
tok = getpass.getpass('Paste your KGAT_ token: ').strip()
pathlib.Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
pathlib.Path('/root/.kaggle/access_token').write_text(tok)
os.chmod('/root/.kaggle/access_token', 0o600)
os.environ['KAGGLE_API_TOKEN'] = tok

!kaggle datasets list -s "breast ultrasound" | head -5

SLUG = 'aryashah2k/breast-ultrasound-images-dataset'
!kaggle datasets download -d {SLUG} -p /content/drive/MyDrive/busi-repro/raw --force

ref                                                              title                                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------------  ------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
aryashah2k/breast-ultrasound-images-dataset                      Breast Ultrasound Images Dataset                   204421470  2021-03-14 04:29:54.023000          74338        505                1  
vuppalaadithyasairam/ultrasound-breast-images-for-breast-cancer  Ultrasound Breast Images for Breast Cancer         591742845  2022-11-03 05:51:50.933000           4875         63             0.75  
orvile/bus-uc-breast-ultrasound                                  BUS_UC - Breast Ultrasound Dataset                 143163924  2025-07-06 00:01:54.773000           1147         36                1  
Datas

In [ ]:
!mkdir -p /content/busi_raw
!unzip -q -o /content/drive/MyDrive/busi-repro/raw/*.zip -d /content/busi_raw
# Sanity check the layout before building anything.
!find /content/busi_raw -maxdepth 3 -type d
!find /content/busi_raw -name '*.png' | wc -l    # expect ~1578 (780 images + masks)

In [ ]:
%cd /content
![ -d busi-repro ] || git clone -q https://github.com/YOUR_USER/busi-repro.git
%cd /content/busi-repro && git pull -q

# --src can point anywhere above the class folders; the script locates them.
!python -m src.build_dataset --src /content/busi_raw --out /content/busi_yolo

**Read the output of that cell before continuing.** You want:
- ~780 kept, split 56.0 / 26.9 / 17.1 percent
- test totals 66 benign / 31 malignant / 20 normal = **117** (matches the paper's Fig. 6)
- train 306 per class after oversampling = 918
- the near-duplicate count, and whether any pair straddles the test boundary

**Copy the printed md5.** It goes in every lane notebook.

In [ ]:
# Publish the processed zip + the reports.
!cp /content/busi_yolo.zip /content/drive/MyDrive/busi-repro/
!cp /content/busi_yolo/manifest.csv /content/busi_yolo/dedup_report.csv \
    /content/drive/MyDrive/busi-repro/
!md5sum /content/busi_yolo.zip
!du -h /content/busi_yolo.zip

In [ ]:
# MedSAM embedding cache for the 117 test images. CPU, ~10 min, no GPU quota.
!pip -q install 'transformers>=4.44' torch --no-warn-conflicts
!python -m src.medsam cache --images /content/busi_yolo/test/images \
    --out /content/drive/MyDrive/busi-repro/medsam_cache

In [ ]:
# Section 4.3 oracle numbers. Independent of YOLO, so it can run right now.
!python -m src.medsam standalone --data /content/busi_yolo \
    --cache /content/drive/MyDrive/busi-repro/medsam_cache \
    --src /content/busi_raw \
    --out /content/drive/MyDrive/busi-repro/results/medsam_standalone.json